In [2]:
import os
from google.colab import userdata
import torch.optim as optim
import wandb
import sys
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torchvision.transforms as T
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import torchvision.models as models


In [3]:

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
os.environ['GITHUB_USERNAME'] = userdata.get('GITHUB_USERNAME')

!git clone https://{os.environ['GITHUB_TOKEN']}@github.com/{os.environ['GITHUB_USERNAME']}/ML_hw_04_Facial-Expression-Recognition-Challenge.git
%cd ML_hw_04_Facial-Expression-Recognition-Challenge

!pip install -q wandb kaggle


wandb.login(key=os.environ['WANDB_API_KEY'])

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(os.environ['KAGGLE_KEY'])

os.makedirs('/content/data', exist_ok=True)
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge -p /content/data
!unzip -q -o /content/data/*.zip -d /content/data

sys.path.append('/content/ML_hw_04_Facial-Expression-Recognition-Challenge/src')
from dataset import FERDataset, load_data, EMOTION_LABELS

print("Setup complete!")

Cloning into 'ML_hw_04_Facial-Expression-Recognition-Challenge'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 98 (delta 49), reused 10 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 300.55 KiB | 10.73 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/ML_hw_04_Facial-Expression-Recognition-Challenge


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kgord23 (kgord23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 285M/285M [00:14<00:00, 20.4MB/s]

Setup complete!


In [4]:
train, val, test = load_data('/content/data/train.csv')
print("Train:", train.shape, "Val:", val.shape, "Test:", test.shape)


class_counts = train['emotion'].value_counts().sort_index().values
sample_weights = 1.0 / class_counts[train['emotion'].values]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])

train_dataset = FERDataset(train, transform=train_transform)
val_dataset = FERDataset(val)
test_dataset = FERDataset(test)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Loaders ready!")

Train: (20095, 2) Val: (4307, 2) Test: (4307, 2)
Loaders ready!


In [ ]:
class ResNetFER(nn.Module):
    def __init__(self, num_classes=7, dropout_p=0.3, freeze_backbone=False):
        super().__init__()
        self.resnet = models.resnet18(pretrained=True)

        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout_p),
            nn.Linear(in_features, num_classes)
        )

        if freeze_backbone:
            for name, param in self.resnet.named_parameters():
                if 'fc' not in name and 'conv1' not in name:
                    param.requires_grad = False

    def forward(self, x):
        return self.resnet(x)

In [5]:
def train_model(model, train_loader, val_loader, config, group_name, run_name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    smoothing = config.get('label_smoothing', 0.0)
    criterion = nn.CrossEntropyLoss(label_smoothing=smoothing)
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'],
                           weight_decay=config.get('weight_decay', 0))

    scheduler = None
    if config.get('scheduler') == 'ReduceLROnPlateau':
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    run = wandb.init(project="facial-expression-recognition", group=group_name, name=run_name, config=config)

    best_val_loss = float('inf')
    patience_counter = 0
    early_stop_patience = config.get('early_stop_patience', 4)
    best_model_state = None

    for epoch in range(config['epochs']):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        current_lr = optimizer.param_groups[0]['lr']
        wandb.log({"epoch": epoch+1, "train_loss": train_loss, "train_acc": train_acc,
                   "val_loss": val_loss, "val_acc": val_acc, "lr": current_lr})
        print(f"Epoch {epoch+1}/{config['epochs']} | Train Loss: {train_loss:.4f}, "
              f"Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, "
              f"Val Acc: {val_acc:.4f} | LR: {current_lr:.6f}")

        if scheduler:
            scheduler.step(val_loss)

        if config.get('early_stopping'):
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_state = model.state_dict()
            else:
                patience_counter += 1
                if patience_counter >= early_stop_patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    model.load_state_dict(best_model_state)
                    break

    wandb.finish()
    return model

In [ ]:
model_check = ResNetFER(freeze_backbone=False)
sample_batch, sample_labels = next(iter(train_loader))

print("Input shape:", sample_batch.shape)
output = model_check(sample_batch)
print("Output shape:", output.shape)

criterion = nn.CrossEntropyLoss()
loss = criterion(output, sample_labels)
print("Loss:", loss.item(), "| Expected ~", torch.log(torch.tensor(7.0)).item())

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 62.1MB/s]


Input shape: torch.Size([64, 1, 48, 48])
Output shape: torch.Size([64, 7])
Loss: 2.367779016494751 | Expected ~ 1.945910096168518


In [ ]:
small_subset = train.sample(n=200, random_state=42)
small_dataset = FERDataset(small_subset, transform=train_transform)
small_loader = DataLoader(small_dataset, batch_size=32, shuffle=True)

config_sanity4 = {
    "architecture": "ResNetFER-sanity",
    "learning_rate": 0.001,
    "batch_size": 32,
    "epochs": 15,
    "optimizer": "Adam",
    "dropout": 0.3,
    "scheduler": "None",
    "early_stopping": False
}

model_sanity4 = ResNetFER(dropout_p=0.3, freeze_backbone=False)
model_sanity4 = train_model(model_sanity4, small_loader, small_loader, config_sanity4,
                             group_name="sanity-checks",
                             run_name="architecture4-resnet-overfit-200")

Epoch 1/15 | Train Loss: 2.5166, Train Acc: 0.1900 | Val Loss: 2.4762, Val Acc: 0.2500 | LR: 0.001000
Epoch 2/15 | Train Loss: 2.2278, Train Acc: 0.3200 | Val Loss: 2.4834, Val Acc: 0.1600 | LR: 0.001000
Epoch 3/15 | Train Loss: 1.9416, Train Acc: 0.3400 | Val Loss: 2.0800, Val Acc: 0.2200 | LR: 0.001000
Epoch 4/15 | Train Loss: 1.7176, Train Acc: 0.4000 | Val Loss: 2.0104, Val Acc: 0.2500 | LR: 0.001000
Epoch 5/15 | Train Loss: 1.6626, Train Acc: 0.4100 | Val Loss: 1.6519, Val Acc: 0.3400 | LR: 0.001000
Epoch 6/15 | Train Loss: 1.3674, Train Acc: 0.5150 | Val Loss: 1.4217, Val Acc: 0.4650 | LR: 0.001000
Epoch 7/15 | Train Loss: 1.1856, Train Acc: 0.5650 | Val Loss: 1.2604, Val Acc: 0.5450 | LR: 0.001000
Epoch 8/15 | Train Loss: 1.4354, Train Acc: 0.5050 | Val Loss: 1.4915, Val Acc: 0.5300 | LR: 0.001000
Epoch 9/15 | Train Loss: 1.3469, Train Acc: 0.5100 | Val Loss: 1.7903, Val Acc: 0.4500 | LR: 0.001000
Epoch 10/15 | Train Loss: 1.3268, Train Acc: 0.5050 | Val Loss: 1.5201, Val Acc: 0

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▃▄▄▅▆▅▅▅▇▇▆▇█
train_loss,█▇▆▅▄▃▂▃▃▃▂▂▂▂▁
val_acc,▂▁▂▂▃▅▆▅▄▅▇▆▇██
val_loss,██▆▆▅▄▃▄▅▄▂▄▂▁▁
epoch,15
lr,0.001
train_acc,0.7
train_loss,0.88957
val_acc,0.755


In [ ]:
config_a4_r1 = {
    "architecture": "ResNet18-FER", "learning_rate": 0.001, "batch_size": 64,
    "epochs": 10, "optimizer": "Adam", "dropout": 0.3,
    "scheduler": "ReduceLROnPlateau", "early_stopping": True, "early_stop_patience": 2,
    "pretrained": True, "freeze_backbone": True,
    "weighted_sampling": False, "augmentation": "none"
}

model_a4_r1 = ResNetFER(dropout_p=0.3, freeze_backbone=True)
model_a4_r1 = train_model(model_a4_r1, train_loader, val_loader, config_a4_r1,
                           group_name="architecture-4-resnet18",
                           run_name="feature-extraction-frozen")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/10 | Train Loss: 1.8086, Train Acc: 0.2993 | Val Loss: 1.5266, Val Acc: 0.4100 | LR: 0.001000
Epoch 2/10 | Train Loss: 1.5044, Train Acc: 0.4282 | Val Loss: 1.5480, Val Acc: 0.4040 | LR: 0.001000
Epoch 3/10 | Train Loss: 1.3518, Train Acc: 0.4897 | Val Loss: 1.3730, Val Acc: 0.4681 | LR: 0.001000
Epoch 4/10 | Train Loss: 1.2470, Train Acc: 0.5280 | Val Loss: 1.3419, Val Acc: 0.4962 | LR: 0.001000
Epoch 5/10 | Train Loss: 1.1849, Train Acc: 0.5470 | Val Loss: 1.4588, Val Acc: 0.4567 | LR: 0.001000
Epoch 6/10 | Train Loss: 1.1227, Train Acc: 0.5766 | Val Loss: 1.5419, Val Acc: 0.4432 | LR: 0.001000
Early stopping at epoch 6


epoch,▁▂▄▅▇█
lr,▁▁▁▁▁▁
train_acc,▁▄▆▇▇█
train_loss,█▅▃▂▂▁
val_acc,▁▁▆█▅▄
val_loss,▇█▂▁▅█
epoch,6
lr,0.001
train_acc,0.57656
train_loss,1.12273
val_acc,0.44323


In [ ]:
config_a4_r2 = {
    "architecture": "ResNet18-FER", "learning_rate": 0.0001, "batch_size": 64,
    "epochs": 20, "optimizer": "Adam", "dropout": 0.3,
    "scheduler": "ReduceLROnPlateau", "early_stopping": True, "early_stop_patience": 4,
    "pretrained": True, "freeze_backbone": False,
    "weighted_sampling": False, "augmentation": "none"
}

model_a4_r2 = ResNetFER(dropout_p=0.3, freeze_backbone=False)
model_a4_r2 = train_model(model_a4_r2, train_loader, val_loader, config_a4_r2,
                           group_name="architecture-4-resnet18",
                           run_name="full-finetune-clean-data")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/20 | Train Loss: 1.9605, Train Acc: 0.2497 | Val Loss: 1.7467, Val Acc: 0.3271 | LR: 0.000100
Epoch 2/20 | Train Loss: 1.6632, Train Acc: 0.3577 | Val Loss: 1.5872, Val Acc: 0.3838 | LR: 0.000100
Epoch 3/20 | Train Loss: 1.5026, Train Acc: 0.4267 | Val Loss: 1.5238, Val Acc: 0.4114 | LR: 0.000100
Epoch 4/20 | Train Loss: 1.3794, Train Acc: 0.4722 | Val Loss: 1.4806, Val Acc: 0.4365 | LR: 0.000100
Epoch 5/20 | Train Loss: 1.2938, Train Acc: 0.5077 | Val Loss: 1.4244, Val Acc: 0.4634 | LR: 0.000100
Epoch 6/20 | Train Loss: 1.2088, Train Acc: 0.5406 | Val Loss: 1.4162, Val Acc: 0.4702 | LR: 0.000100
Epoch 7/20 | Train Loss: 1.1440, Train Acc: 0.5631 | Val Loss: 1.3931, Val Acc: 0.4818 | LR: 0.000100
Epoch 8/20 | Train Loss: 1.0811, Train Acc: 0.5918 | Val Loss: 1.3750, Val Acc: 0.4999 | LR: 0.000100
Epoch 9/20 | Train Loss: 1.0218, Train Acc: 0.6138 | Val Loss: 1.3557, Val Acc: 0.5082 | LR: 0.000100
Epoch 10/20 | Train Loss: 0.9730, Train Acc: 0.6335 | Val Loss: 1.3331, Val Acc: 0

epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
lr,███████████████▁
train_acc,▁▂▃▄▅▅▅▆▆▆▆▇▇▇▇█
train_loss,█▆▆▅▄▄▄▃▃▃▃▂▂▂▂▁
val_acc,▁▃▄▄▅▅▆▆▇▇▇▇▇▇▇█
val_loss,█▅▄▄▃▂▂▂▁▁▁▁▂▃▂▂
epoch,16
lr,5e-05
train_acc,0.76233
train_loss,0.64825
val_acc,0.55514


In [ ]:
config_a4_r3 = {
    "architecture": "ResNet18-FER",
    "learning_rate": 0.0001,
    "batch_size": 64,
    "epochs": 25,
    "optimizer": "Adam",
    "dropout": 0.3,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping": True,
    "early_stop_patience": 4,
    "pretrained": True,
    "freeze_backbone": False,
    "weighted_sampling": True,
    "augmentation": "flip+rotation+translation"
}

model_a4_r3 = ResNetFER(dropout_p=0.3, freeze_backbone=False)
model_a4_r3 = train_model(
    model_a4_r3,
    train_loader,
    val_loader,
    config_a4_r3,
    group_name="architecture-4-resnet18",
    run_name="full-finetune-with-augmented-sampler"
)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/25 | Train Loss: 1.9824, Train Acc: 0.2424 | Val Loss: 1.8059, Val Acc: 0.2907 | LR: 0.000100
Epoch 2/25 | Train Loss: 1.6646, Train Acc: 0.3555 | Val Loss: 1.6263, Val Acc: 0.3668 | LR: 0.000100
Epoch 3/25 | Train Loss: 1.4933, Train Acc: 0.4281 | Val Loss: 1.5017, Val Acc: 0.4219 | LR: 0.000100
Epoch 4/25 | Train Loss: 1.3606, Train Acc: 0.4790 | Val Loss: 1.4369, Val Acc: 0.4560 | LR: 0.000100
Epoch 5/25 | Train Loss: 1.2451, Train Acc: 0.5243 | Val Loss: 1.4116, Val Acc: 0.4595 | LR: 0.000100
Epoch 6/25 | Train Loss: 1.1944, Train Acc: 0.5440 | Val Loss: 1.3593, Val Acc: 0.4918 | LR: 0.000100
Epoch 7/25 | Train Loss: 1.1286, Train Acc: 0.5721 | Val Loss: 1.3652, Val Acc: 0.4834 | LR: 0.000100
Epoch 8/25 | Train Loss: 1.0653, Train Acc: 0.5994 | Val Loss: 1.3523, Val Acc: 0.4990 | LR: 0.000100
Epoch 9/25 | Train Loss: 1.0220, Train Acc: 0.6166 | Val Loss: 1.3371, Val Acc: 0.5055 | LR: 0.000100
Epoch 10/25 | Train Loss: 0.9508, Train Acc: 0.6443 | Val Loss: 1.3600, Val Acc: 0

epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
lr,████████████▃▃▃▃▁
train_acc,▁▂▃▄▅▅▅▆▆▆▆▇▇▇███
train_loss,█▆▆▅▄▄▄▃▃▃▃▂▂▂▁▁▁
val_acc,▁▃▄▅▅▆▆▇▇▇▇▇██▇▇█
val_loss,█▅▃▃▂▁▁▁▁▁▁▂▁▂▂▂▂
epoch,17
lr,3e-05
train_acc,0.79268
train_loss,0.578
val_acc,0.55352


In [ ]:
config_a4_r4 = {
    "architecture": "ResNet18-FER",
    "learning_rate": 0.0001,
    "batch_size": 64,
    "epochs": 25,
    "optimizer": "Adam",
    "dropout": 0.3,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping": True,
    "early_stop_patience": 4,
    "pretrained": True,
    "freeze_backbone": True,
    "weighted_sampling": True,
    "augmentation": "flip+rotation+translation"
}

model_a4_r4 = ResNetFER(
    dropout_p=0.3,
    freeze_backbone=True
)

model_a4_r4 = train_model(
    model_a4_r4,
    train_loader,
    val_loader,
    config_a4_r4,
    group_name="architecture-4-resnet18",
    run_name="frozen-backbone-with-augmented-sampler"
)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


Epoch 1/25 | Train Loss: 2.0613, Train Acc: 0.1975 | Val Loss: 1.9052, Val Acc: 0.2208 | LR: 0.000100
Epoch 2/25 | Train Loss: 1.8290, Train Acc: 0.2723 | Val Loss: 1.7842, Val Acc: 0.2979 | LR: 0.000100
Epoch 3/25 | Train Loss: 1.6974, Train Acc: 0.3403 | Val Loss: 1.6892, Val Acc: 0.3432 | LR: 0.000100
Epoch 4/25 | Train Loss: 1.5798, Train Acc: 0.3952 | Val Loss: 1.6127, Val Acc: 0.3826 | LR: 0.000100
Epoch 5/25 | Train Loss: 1.4845, Train Acc: 0.4301 | Val Loss: 1.5648, Val Acc: 0.4137 | LR: 0.000100
Epoch 6/25 | Train Loss: 1.4048, Train Acc: 0.4621 | Val Loss: 1.5065, Val Acc: 0.4370 | LR: 0.000100
Epoch 7/25 | Train Loss: 1.3372, Train Acc: 0.4915 | Val Loss: 1.4549, Val Acc: 0.4402 | LR: 0.000100
Epoch 8/25 | Train Loss: 1.2855, Train Acc: 0.5089 | Val Loss: 1.4576, Val Acc: 0.4479 | LR: 0.000100
Epoch 9/25 | Train Loss: 1.2256, Train Acc: 0.5280 | Val Loss: 1.3975, Val Acc: 0.4741 | LR: 0.000100
Epoch 10/25 | Train Loss: 1.1826, Train Acc: 0.5493 | Val Loss: 1.3829, Val Acc: 0

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████████████████████▁▁▁▁▁
train_acc,▁▂▃▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
train_loss,█▇▆▆▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
val_acc,▁▃▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
val_loss,█▇▅▄▄▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,25
lr,5e-05
train_acc,0.74078
train_loss,0.70893
val_acc,0.54214


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNetFER_CustomInput(nn.Module):
    def __init__(self, num_classes=7, dropout_p=0.3, freeze_backbone=False):
        super(ResNetFER_CustomInput, self).__init__()


        weights = models.ResNet18_Weights.DEFAULT
        self.backbone = models.resnet18(weights=weights)


        self.backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )


        self.backbone.maxpool = nn.Identity()


        if freeze_backbone:
            for name, param in self.backbone.named_parameters():

                if "conv1" not in name:
                    param.requires_grad = False
        else:
            for param in self.backbone.parameters():
                param.requires_grad = True


        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
config_a4_r5 = {
    "architecture": "ResNet18-CustomInput-FER",
    "learning_rate": 0.0001,
    "batch_size": 64,
    "epochs": 25,
    "optimizer": "Adam",
    "dropout": 0.3,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping": True,
    "early_stop_patience": 4,
    "pretrained": True,
    "freeze_backbone": False,
    "weighted_sampling": True,
    "augmentation": "flip+rotation+translation"
}


model_a4_r5 = ResNetFER_CustomInput(dropout_p=0.3, freeze_backbone=False)

model_a4_r5 = train_model(
    model_a4_r5,
    train_loader,
    val_loader,
    config_a4_r5,
    group_name="architecture-4-resnet18",
    run_name="custom-entry-flow-full-pipeline"
)

Epoch 1/25 | Train Loss: 1.3984, Train Acc: 0.4691 | Val Loss: 1.1896, Val Acc: 0.5405 | LR: 0.000100
Epoch 2/25 | Train Loss: 1.0445, Train Acc: 0.6075 | Val Loss: 1.0796, Val Acc: 0.5925 | LR: 0.000100
Epoch 3/25 | Train Loss: 0.9034, Train Acc: 0.6662 | Val Loss: 1.0244, Val Acc: 0.6274 | LR: 0.000100
Epoch 4/25 | Train Loss: 0.8320, Train Acc: 0.6913 | Val Loss: 1.0364, Val Acc: 0.6218 | LR: 0.000100
Epoch 5/25 | Train Loss: 0.7638, Train Acc: 0.7187 | Val Loss: 1.0311, Val Acc: 0.6220 | LR: 0.000100
Epoch 6/25 | Train Loss: 0.7268, Train Acc: 0.7335 | Val Loss: 1.0450, Val Acc: 0.6294 | LR: 0.000100
Epoch 7/25 | Train Loss: 0.6321, Train Acc: 0.7722 | Val Loss: 1.0029, Val Acc: 0.6501 | LR: 0.000050
Epoch 8/25 | Train Loss: 0.5787, Train Acc: 0.7911 | Val Loss: 1.0210, Val Acc: 0.6545 | LR: 0.000050
Epoch 9/25 | Train Loss: 0.5286, Train Acc: 0.8137 | Val Loss: 1.0630, Val Acc: 0.6455 | LR: 0.000050
Epoch 10/25 | Train Loss: 0.4965, Train Acc: 0.8233 | Val Loss: 1.0583, Val Acc: 0

epoch,▁▂▂▃▄▅▅▆▇▇█
lr,██████▃▃▃▃▁
train_acc,▁▄▅▅▆▆▇▇▇██
train_loss,█▅▄▄▃▃▂▂▂▁▁
val_acc,▁▄▆▆▆▆██▇██
val_loss,█▄▂▂▂▃▁▂▃▃▄
epoch,11
lr,3e-05
train_acc,0.84315
train_loss,0.44412
val_acc,0.65428


In [6]:
train_transform_v2 = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.RandomErasing(p=0.3, scale=(0.02, 0.1)),
])


train_dataset_v2 = FERDataset(train, transform=train_transform_v2)
train_loader_v2 = DataLoader(train_dataset_v2, batch_size=64, sampler=sampler)

In [ ]:

config_a4_r6 = {
    "architecture": "ResNet18-CustomInput-FER",
    "learning_rate": 0.0001,
    "batch_size": 64,
    "epochs": 25,
    "optimizer": "Adam",
    "dropout": 0.5,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping": True,
    "early_stop_patience": 4,
    "pretrained": True,
    "freeze_backbone": False,
    "weighted_sampling": True,
    "augmentation": "flip+rotation15+translation0.1+erasing"
}

model_a4_r6 = ResNetFER_CustomInput(dropout_p=0.5, freeze_backbone=False)
model_a4_r6 = train_model(
    model_a4_r6, train_loader_v2, val_loader, config_a4_r6,
    group_name="architecture-4-resnet18",
    run_name="stronger-regularization-dropout0.5-wd1e4"
)

Epoch 1/25 | Train Loss: 1.5517, Train Acc: 0.4013 | Val Loss: 1.2714, Val Acc: 0.5210 | LR: 0.000100
Epoch 2/25 | Train Loss: 1.1850, Train Acc: 0.5593 | Val Loss: 1.0719, Val Acc: 0.5907 | LR: 0.000100
Epoch 3/25 | Train Loss: 1.0683, Train Acc: 0.6008 | Val Loss: 1.0442, Val Acc: 0.6058 | LR: 0.000100
Epoch 4/25 | Train Loss: 0.9792, Train Acc: 0.6393 | Val Loss: 1.0475, Val Acc: 0.6074 | LR: 0.000100
Epoch 5/25 | Train Loss: 0.9242, Train Acc: 0.6573 | Val Loss: 1.0224, Val Acc: 0.6092 | LR: 0.000100
Epoch 6/25 | Train Loss: 0.8709, Train Acc: 0.6776 | Val Loss: 0.9999, Val Acc: 0.6366 | LR: 0.000100
Epoch 7/25 | Train Loss: 0.8405, Train Acc: 0.6928 | Val Loss: 1.0362, Val Acc: 0.6306 | LR: 0.000100
Epoch 8/25 | Train Loss: 0.7861, Train Acc: 0.7084 | Val Loss: 1.0030, Val Acc: 0.6387 | LR: 0.000100
Epoch 9/25 | Train Loss: 0.7632, Train Acc: 0.7188 | Val Loss: 0.9983, Val Acc: 0.6424 | LR: 0.000100
Epoch 10/25 | Train Loss: 0.7258, Train Acc: 0.7338 | Val Loss: 1.0248, Val Acc: 0

epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
lr,███████████████▁
train_acc,▁▄▄▅▅▆▆▆▆▇▇▇▇▇▇█
train_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▁▁
val_acc,▁▄▅▅▅▇▆▇▇▇▇▇▇█▇█
val_loss,█▃▂▂▂▁▂▁▁▂▂▁▂▂▃▃
epoch,16
lr,5e-05
train_acc,0.81254
train_loss,0.52159
val_acc,0.66775


In [7]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNetFER_AdvancedRegularized(nn.Module):
    def __init__(self, num_classes=7, dropout_p=0.5):
        super(ResNetFER_AdvancedRegularized, self).__init__()

        weights = models.ResNet18_Weights.DEFAULT
        self.backbone = models.resnet18(weights=weights)


        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()


        self.feature_dropout = nn.Dropout2d(p=0.15)


        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.feature_dropout(x)

        x = self.backbone.layer3(x)
        x = self.feature_dropout(x)

        x = self.backbone.layer4(x)


        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.backbone.fc(x)
        return x

In [8]:
config_a4_r7 = {
    "architecture": "ResNet18-AdvancedReg-FER",
    "learning_rate": 0.0001,
    "batch_size": 64,
    "epochs": 25,
    "optimizer": "Adam",
    "dropout": 0.5,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping": True,
    "early_stop_patience": 4,
    "pretrained": True,
    "freeze_backbone": False,
    "weighted_sampling": True,
    "augmentation": "flip+rotation15+translation0.1+erasing"
}


model_a4_r7 = ResNetFER_AdvancedRegularized(dropout_p=0.5)

model_a4_r7 = train_model(
    model_a4_r7,
    train_loader_v2,
    val_loader,
    config_a4_r7,
    group_name="architecture-4-resnet18",
    run_name="internal-dropout-with-label-smoothing"
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 192MB/s]


Epoch 1/25 | Train Loss: 1.8092, Train Acc: 0.2874 | Val Loss: 1.5539, Val Acc: 0.4393 | LR: 0.000100
Epoch 2/25 | Train Loss: 1.5355, Train Acc: 0.4595 | Val Loss: 1.3809, Val Acc: 0.5250 | LR: 0.000100
Epoch 3/25 | Train Loss: 1.4089, Train Acc: 0.5314 | Val Loss: 1.3187, Val Acc: 0.5686 | LR: 0.000100
Epoch 4/25 | Train Loss: 1.3418, Train Acc: 0.5649 | Val Loss: 1.2892, Val Acc: 0.5876 | LR: 0.000100
Epoch 5/25 | Train Loss: 1.2965, Train Acc: 0.5842 | Val Loss: 1.2501, Val Acc: 0.5997 | LR: 0.000100
Epoch 6/25 | Train Loss: 1.2476, Train Acc: 0.6122 | Val Loss: 1.2427, Val Acc: 0.6055 | LR: 0.000100
Epoch 7/25 | Train Loss: 1.2240, Train Acc: 0.6246 | Val Loss: 1.2038, Val Acc: 0.6292 | LR: 0.000100
Epoch 8/25 | Train Loss: 1.1914, Train Acc: 0.6434 | Val Loss: 1.1958, Val Acc: 0.6371 | LR: 0.000100
Epoch 9/25 | Train Loss: 1.1627, Train Acc: 0.6585 | Val Loss: 1.1774, Val Acc: 0.6394 | LR: 0.000100
Epoch 10/25 | Train Loss: 1.1470, Train Acc: 0.6608 | Val Loss: 1.1898, Val Acc: 0

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,██████████████▃▃▃▃▃▃▁▁▁▁▁
train_acc,▁▃▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████
train_loss,█▆▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
val_acc,▁▃▅▅▆▆▆▇▇▇▇▇▇▇███████████
val_loss,█▅▄▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch,25
lr,3e-05
train_acc,0.77577
train_loss,0.93016
val_acc,0.68006


In [ ]:
train_transform_v3 = T.Compose([
    T.Resize((112, 112)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.RandomErasing(p=0.3, scale=(0.02, 0.1)),
])


val_transform_v3 = T.Compose([
    T.Resize((112, 112)),
])


train_dataset_v3 = FERDataset(train, transform=train_transform_v3)
val_dataset_v3 = FERDataset(val, transform=val_transform_v3)


train_loader_v3 = DataLoader(train_dataset_v3, batch_size=64, sampler=sampler)
val_loader_v3 = DataLoader(val_dataset_v3, batch_size=64, shuffle=False)

In [ ]:
config_a4_r8 = {
    "architecture": "ResNet18-AdvancedReg-112x112",
    "learning_rate": 0.0001,
    "batch_size": 64,
    "epochs": 25,
    "optimizer": "Adam",
    "dropout": 0.5,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping": True,
    "early_stop_patience": 4,
    "pretrained": True,
    "freeze_backbone": False,
    "weighted_sampling": True,
    "augmentation": "resize112+flip+rotation15+translation0.1+erasing"
}


model_a4_r8 = ResNetFER_AdvancedRegularized(dropout_p=0.5)
model_a4_r8 = train_model(
    model_a4_r8,
    train_loader_v3,
    val_loader_v3,
    config_a4_r8,
    group_name="architecture-4-resnet18",
    run_name="final-grand-finale-112x112"
)

Epoch 1/25 | Train Loss: 1.8081, Train Acc: 0.2899 | Val Loss: 1.5478, Val Acc: 0.4363 | LR: 0.000100
Epoch 2/25 | Train Loss: 1.5054, Train Acc: 0.4781 | Val Loss: 1.3845, Val Acc: 0.5196 | LR: 0.000100
Epoch 3/25 | Train Loss: 1.3695, Train Acc: 0.5504 | Val Loss: 1.3086, Val Acc: 0.5728 | LR: 0.000100
Epoch 4/25 | Train Loss: 1.2997, Train Acc: 0.5901 | Val Loss: 1.2657, Val Acc: 0.6013 | LR: 0.000100
Epoch 5/25 | Train Loss: 1.2462, Train Acc: 0.6191 | Val Loss: 1.2433, Val Acc: 0.6083 | LR: 0.000100
Epoch 6/25 | Train Loss: 1.2201, Train Acc: 0.6324 | Val Loss: 1.2231, Val Acc: 0.6229 | LR: 0.000100
Epoch 7/25 | Train Loss: 1.1818, Train Acc: 0.6513 | Val Loss: 1.1973, Val Acc: 0.6306 | LR: 0.000100
Epoch 8/25 | Train Loss: 1.1579, Train Acc: 0.6676 | Val Loss: 1.1891, Val Acc: 0.6494 | LR: 0.000100
Epoch 9/25 | Train Loss: 1.1363, Train Acc: 0.6742 | Val Loss: 1.1961, Val Acc: 0.6464 | LR: 0.000100
Epoch 10/25 | Train Loss: 1.1073, Train Acc: 0.6914 | Val Loss: 1.1678, Val Acc: 0

epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
lr,████████████████████▁
train_acc,▁▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇███
train_loss,█▆▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
val_acc,▁▃▅▆▆▆▇▇▇▇▇▇▇▇▇▇█▇███
val_loss,█▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▂▁
epoch,21
lr,5e-05
train_acc,0.80199
train_loss,0.89777
val_acc,0.68191


In [9]:
def evaluate_on_unseen_test_set(model, test_loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)


    model.eval()


    criterion = nn.CrossEntropyLoss()

    test_loss, test_correct, test_total = 0, 0, 0


    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            test_correct += (predicted == labels).sum().item()
            test_total += labels.size(0)

    final_test_loss = test_loss / test_total
    final_test_acc = test_correct / test_total

    print("=" * 50)
    print(f"🔒 FINAL EVALUATION ON UNSEEN TEST SET (Run 4.7 Winner)")
    print("=" * 50)
    print(f"📉 Test Loss: {final_test_loss:.4f}")
    print(f"🎯 Test Accuracy: {final_test_acc * 100:.2f}%")
    print("=" * 50)

    return final_test_loss, final_test_acc

# გაშვება (გადაეცი შენი ნავარჯიშები საუკეთესო მოდელი 4.7 რანიდან და შენი მზა test_loader)
final_loss, final_acc = evaluate_on_unseen_test_set(model_a4_r7, test_loader)

🔒 FINAL EVALUATION ON UNSEEN TEST SET (Run 4.7 Winner)
📉 Test Loss: 0.8923
🎯 Test Accuracy: 67.91%
